In [113]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import re
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rawan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rawan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [114]:
df = pd.read_csv('CompanyReviews.csv')

In [115]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          40046 non-null  int64 
 1   review_description  40045 non-null  object
 2   rating              40046 non-null  int64 
 3   company             40046 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.2+ MB


In [116]:
df.head()

,Unnamed: 0,review_description,rating,company
0,0,رائع,1,talbat
1,1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,1,talbat
2,2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,-1,talbat
3,3,لماذا لا يمكننا طلب من ماكدونالدز؟,-1,talbat
4,4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,-1,talbat


In [117]:
null_counts = df.isnull().sum()

print(null_counts)
df.dropna(inplace=True)
null_counts = df.isnull().sum()
print(null_counts)

Unnamed: 0            0
review_description    1
rating                0
company               0
dtype: int64
Unnamed: 0            0
review_description    0
rating                0
company               0
dtype: int64


In [118]:
def normalize_arabic(text):
    # if pd.isna(text):
    #     return ""
    text = re.sub(r"[إأآا]", "ا", text)    # Normalize various forms of 'alef'
    text = re.sub(r"ى", "ي", text)        # Normalize 'alef maqsoora' to 'ya'
    text = re.sub(r"ؤ", "ء", text)        # Normalize 'waw with hamza' to 'hamza'
    #text = re.sub(r"ئ", "ء", text)        # Normalize 'ya with hamza' to 'hamza'
    text = re.sub(r"ة", "ه", text)        # Normalize 'ta marbuta' to 'ha'
    text = re.sub(r"گ", "ك", text)        # Normalize 'kaf with dot' to 'kaf'
    text = re.sub(r"ڤ", "ف", text)        # Normalize 'v' to 'fa'
    text = re.sub(r"چ", "ج", text)        # Normalize 'che' to 'jeem'
    text = re.sub(r"پ", "ب", text)        # Normalize 'pe' to 'be'
    text = re.sub(r"ڜ", "ش", text)        # Normalize 'sh' to 'sheen'
    text = re.sub(r"ڪ", "ك", text)        # Normalize 'kaf with dot' to 'kaf'
    text = re.sub(r"ڧ", "ق", text)        # Normalize 'fe' to 'qaf'
    text = re.sub(r"ٱ", "ا", text)        # Normalize 'alef with wasla' to 'alef'
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)       # Remove tashkeel (diacritics)
    text = re.sub(r"\s+", " ", text).strip()# Remove extra spaces
   
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)# Remove punctuation & symbols

    
    return text


In [119]:
df['review_description'] = df['review_description'].apply(normalize_arabic)


In [120]:
from nltk.tokenize import wordpunct_tokenize

df['tokens'] = df['review_description'].apply(
    lambda x: wordpunct_tokenize(x) if isinstance(x, str) and x.strip() else []
)


In [121]:
df

,Unnamed: 0,review_description,rating,company,tokens
0,0,رائع,1,talbat,[رائع]
1,1,برنامج رائع جدا يساعد علي تلبيه الاحتياجات بشك...,1,talbat,"[برنامج, رائع, جدا, يساعد, علي, تلبيه, الاحتيا..."
2,2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,-1,talbat,"[التطبيق, لا, يغتح, دائما, بيعطيني, لا, يوجد, ..."
3,3,لماذا لا يمكننا طلب من ماكدونالدز؟,-1,talbat,"[لماذا, لا, يمكننا, طلب, من, ماكدونالدز, ؟]"
4,4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,-1,talbat,"[البرنامج, بيظهر, كل, المطاعم, و, مغلقه, مع, ا..."
...,...,...,...,...,...
40041,128,تجربه جيده بس ينقصها عدم اهتمام خدمه العملاء ب...,0,swvl,"[تجربه, جيده, بس, ينقصها, عدم, اهتمام, خدمه, ا..."
40042,129,انا ساكنه بمنطقه الكينج ولا توجد عربيات قبل ال...,-1,swvl,"[انا, ساكنه, بمنطقه, الكينج, ولا, توجد, عربيات..."
40043,130,جيد ولكن لماذا لا توجد خطوط كثيره من المريوطيه...,0,swvl,"[جيد, ولكن, لماذا, لا, توجد, خطوط, كثيره, من, ..."
40044,131,جيدا جدا ولكن الاسعار عاليه جدا,0,swvl,"[جيدا, جدا, ولكن, الاسعار, عاليه, جدا]"


In [122]:
from nltk.corpus import stopwords
nltk.download('stopwords')

arabic_stopwords = set(stopwords.words('arabic'))

df['tokens'] = df['tokens'].apply(
    lambda tokens: [w for w in tokens if w not in arabic_stopwords]
)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rawan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [123]:
df

,Unnamed: 0,review_description,rating,company,tokens
0,0,رائع,1,talbat,[رائع]
1,1,برنامج رائع جدا يساعد علي تلبيه الاحتياجات بشك...,1,talbat,"[برنامج, رائع, جدا, يساعد, علي, تلبيه, الاحتيا..."
2,2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,-1,talbat,"[التطبيق, يغتح, دائما, بيعطيني, يوجد, اتصال, ب..."
3,3,لماذا لا يمكننا طلب من ماكدونالدز؟,-1,talbat,"[لماذا, يمكننا, طلب, ماكدونالدز, ؟]"
4,4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,-1,talbat,"[البرنامج, بيظهر, المطاعم, مغلقه, انها, بتكون,..."
...,...,...,...,...,...
40041,128,تجربه جيده بس ينقصها عدم اهتمام خدمه العملاء ب...,0,swvl,"[تجربه, جيده, ينقصها, عدم, اهتمام, خدمه, العمل..."
40042,129,انا ساكنه بمنطقه الكينج ولا توجد عربيات قبل ال...,-1,swvl,"[انا, ساكنه, بمنطقه, الكينج, توجد, عربيات, الس..."
40043,130,جيد ولكن لماذا لا توجد خطوط كثيره من المريوطيه...,0,swvl,"[جيد, لماذا, توجد, خطوط, كثيره, المريوطيه, فيص..."
40044,131,جيدا جدا ولكن الاسعار عاليه جدا,0,swvl,"[جيدا, جدا, الاسعار, عاليه, جدا]"


In [124]:
print(arabic_stopwords)

{'خمسة', 'ثماني', 'إليكَ', 'الألاء', 'طاء', 'أم', 'بس', 'ليس', 'أيّ', 'ذهب', 'وا', 'مائة', 'لدن', 'تلقاء', 'سمعا', 'نوفمبر', 'إليكم', 'من', 'راء', 'أين', 'لمّا', 'وجد', 'ذلكن', 'ثلاثون', 'استحال', 'ثلاثمائة', 'اللواتي', 'حاشا', 'آناء', 'ارتدّ', 'ا', 'هَؤلاء', 'كأيّ', 'واهاً', 'كيفما', 'ثاء', 'إياي', 'تجاه', 'ته', 'مثل', 'حاء', 'ليسوا', 'بَلْهَ', 'أنتن', 'تانِك', 'يونيو', 'لوما', 'خلافا', 'لستما', 'إي', 'هاتين', 'لن', 'رويدك', 'ما انفك', 'مايو', 'لي', 'زاي', 'عن', 'حاي', 'ذه', 'صار', 'تلكم', 'خال', 'تبدّل', 'لكنما', 'ليست', 'كذلك', 'هيهات', 'إما', 'أعطى', 'منذ', 'رابع', 'حبيب', 'صبر', 'غداة', 'أنتم', 'هَذَيْنِ', 'تخذ', 'قلما', 'إزاء', 'بَسْ', 'ذو', 'دواليك', 'بهن', 'تسع', 'إى', 'ض', 'إليكن', 'بل', 'جويلية', 'تِه', 'جمعة', 'قطّ', 'هنا', 'لكم', 'ترك', 'سبتمبر', 'شباط', 'الذي', 'خمسون', 'حيثما', 'حيث', 'مارس', 'لكما', 'ذانك', 'ألفى', 'عَدَسْ', 'هل', 'سين', 'خمسمائة', 'ثامن', 'كل', 'كم', 'نفس', 'مما', 'وإذا', 'حَذارِ', 'مافتئ', 'كأين', 'سوى', 'ثلاثمئة', 'إن', 'ثمّة', 'لام', 'فلا', 'شين', 'د

In [125]:
df['label'] = (df['rating'] >= 0).astype(int)
df

,Unnamed: 0,review_description,rating,company,tokens,label
0,0,رائع,1,talbat,[رائع],1
1,1,برنامج رائع جدا يساعد علي تلبيه الاحتياجات بشك...,1,talbat,"[برنامج, رائع, جدا, يساعد, علي, تلبيه, الاحتيا...",1
2,2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,-1,talbat,"[التطبيق, يغتح, دائما, بيعطيني, يوجد, اتصال, ب...",0
3,3,لماذا لا يمكننا طلب من ماكدونالدز؟,-1,talbat,"[لماذا, يمكننا, طلب, ماكدونالدز, ؟]",0
4,4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,-1,talbat,"[البرنامج, بيظهر, المطاعم, مغلقه, انها, بتكون,...",0
...,...,...,...,...,...,...
40041,128,تجربه جيده بس ينقصها عدم اهتمام خدمه العملاء ب...,0,swvl,"[تجربه, جيده, ينقصها, عدم, اهتمام, خدمه, العمل...",1
40042,129,انا ساكنه بمنطقه الكينج ولا توجد عربيات قبل ال...,-1,swvl,"[انا, ساكنه, بمنطقه, الكينج, توجد, عربيات, الس...",0
40043,130,جيد ولكن لماذا لا توجد خطوط كثيره من المريوطيه...,0,swvl,"[جيد, لماذا, توجد, خطوط, كثيره, المريوطيه, فيص...",1
40044,131,جيدا جدا ولكن الاسعار عاليه جدا,0,swvl,"[جيدا, جدا, الاسعار, عاليه, جدا]",1


In [126]:
X_text = df['review_description'].values
y = df['label'].values


In [ ]:


max_words = 20000   
max_len = 100       

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_text)

sequences = tokenizer.texts_to_sequences(X_text)
X = pad_sequences(sequences, maxlen=max_len, padding='post')


In [128]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [129]:
from tensorflow.keras.layers import Embedding

embedding_dim = 128

embedding_layer = Embedding(
    input_dim=max_words,
    output_dim=embedding_dim,
    input_length=max_len
)


c:\Users\rawan\miniconda3\envs\newml\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [130]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

def build_rnn():
    model = Sequential([
        embedding_layer,
        SimpleRNN(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


In [131]:
from tensorflow.keras.layers import LSTM

def build_lstm():
    model = Sequential([
        embedding_layer,
        LSTM(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


In [132]:
from tensorflow.keras.layers import GRU

def build_gru():
    model = Sequential([
        embedding_layer,
        GRU(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


In [133]:
def compile_and_train(model):
    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=10,
        batch_size=32,
        validation_split=0.1,
        verbose=1
    )
    return history


In [135]:
rnn_model = build_rnn()
rnn_hist = compile_and_train(rnn_model)

Epoch 1/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 26s 27ms/step - accuracy: 0.8160 - loss: 0.4494 - val_accuracy: 0.8230 - val_loss: 0.4545
Epoch 2/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 27s 30ms/step - accuracy: 0.8580 - loss: 0.3788 - val_accuracy: 0.8252 - val_loss: 0.4406
Epoch 3/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 28s 31ms/step - accuracy: 0.8610 - loss: 0.3745 - val_accuracy: 0.8224 - val_loss: 0.4518
Epoch 4/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 27s 30ms/step - accuracy: 0.8682 - loss: 0.3637 - val_accuracy: 0.8215 - val_loss: 0.4492
Epoch 5/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 29s 32ms/step - accuracy: 0.8070 - loss: 0.4656 - val_accuracy: 0.7949 - val_loss: 0.5018
Epoch 6/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 26s 29ms/step - accuracy: 0.8265 - loss: 0.4494 - val_accuracy: 0.7949 - val_loss: 0.4960
Epoch 7/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 25s 28ms/step - accuracy: 0.8267 - loss: 0.4481 - val_accuracy: 0.7921 - val_loss: 0.4998
Epoch 8/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 27s 30ms/step - accuracy: 0.8133 - loss: 0.4558 - 

In [136]:
lstm_model = build_lstm()
lstm_hist = compile_and_train(lstm_model)


Epoch 1/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 38s 39ms/step - accuracy: 0.6452 - loss: 0.6534 - val_accuracy: 0.6492 - val_loss: 0.6483
Epoch 2/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 34s 37ms/step - accuracy: 0.6461 - loss: 0.6521 - val_accuracy: 0.6492 - val_loss: 0.6482
Epoch 3/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.6410 - loss: 0.6322 - val_accuracy: 0.6492 - val_loss: 0.6032
Epoch 4/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 30s 34ms/step - accuracy: 0.6438 - loss: 0.6064 - val_accuracy: 0.6492 - val_loss: 0.6008
Epoch 5/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 29s 32ms/step - accuracy: 0.6449 - loss: 0.6059 - val_accuracy: 0.6492 - val_loss: 0.6004
Epoch 6/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 34s 38ms/step - accuracy: 0.6977 - loss: 0.5887 - val_accuracy: 0.6492 - val_loss: 0.6483
Epoch 7/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 38s 42ms/step - accuracy: 0.6454 - loss: 0.6510 - val_accuracy: 0.6492 - val_loss: 0.6482
Epoch 8/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 32s 36ms/step - accuracy: 0.6454 - loss: 0.6508 - 

In [138]:
gru_model = build_gru()
gru_hist = compile_and_train(gru_model)


Epoch 1/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 36s 38ms/step - accuracy: 0.7623 - loss: 0.4831 - val_accuracy: 0.8699 - val_loss: 0.3134
Epoch 2/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 31s 34ms/step - accuracy: 0.9128 - loss: 0.2420 - val_accuracy: 0.8764 - val_loss: 0.3142
Epoch 3/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.9352 - loss: 0.1918 - val_accuracy: 0.8611 - val_loss: 0.3462
Epoch 4/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 33s 37ms/step - accuracy: 0.9483 - loss: 0.1546 - val_accuracy: 0.8571 - val_loss: 0.3847
Epoch 5/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 38s 42ms/step - accuracy: 0.9573 - loss: 0.1252 - val_accuracy: 0.8546 - val_loss: 0.4265
Epoch 6/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.9629 - loss: 0.1079 - val_accuracy: 0.8514 - val_loss: 0.5175
Epoch 7/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 34s 38ms/step - accuracy: 0.9666 - loss: 0.0949 - val_accuracy: 0.8524 - val_loss: 0.6193
Epoch 8/10
901/901 ━━━━━━━━━━━━━━━━━━━━ 34s 38ms/step - accuracy: 0.9694 - loss: 0.0857 - 

In [139]:
rnn_acc = rnn_model.evaluate(X_test, y_test, verbose=0)[1]
lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)[1]
gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)[1]

print(f"RNN Accuracy  : {rnn_acc:.4f}")
print(f"LSTM Accuracy : {lstm_acc:.4f}")
print(f"GRU Accuracy  : {gru_acc:.4f}")


RNN Accuracy  : 0.7776
LSTM Accuracy : 0.8575
GRU Accuracy  : 0.8472


In [141]:
from sklearn.metrics import classification_report
import numpy as np


In [142]:
# RNN
y_pred_rnn = rnn_model.predict(X_test)
y_pred_rnn = (y_pred_rnn > 0.5).astype(int)

# LSTM
y_pred_lstm = lstm_model.predict(X_test)
y_pred_lstm = (y_pred_lstm > 0.5).astype(int)

# GRU
y_pred_gru = gru_model.predict(X_test)
y_pred_gru = (y_pred_gru > 0.5).astype(int)


251/251 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step


In [143]:
from sklearn.metrics import precision_score, recall_score, f1_score

# RNN
rnn_precision = precision_score(y_test, y_pred_rnn)
rnn_recall = recall_score(y_test, y_pred_rnn)
rnn_f1 = f1_score(y_test, y_pred_rnn)

# LSTM
lstm_precision = precision_score(y_test, y_pred_lstm)
lstm_recall = recall_score(y_test, y_pred_lstm)
lstm_f1 = f1_score(y_test, y_pred_lstm)

# GRU
gru_precision = precision_score(y_test, y_pred_gru)
gru_recall = recall_score(y_test, y_pred_gru)
gru_f1 = f1_score(y_test, y_pred_gru)

print(f"RNN → Precision: {rnn_precision:.4f}, Recall: {rnn_recall:.4f}, F1: {rnn_f1:.4f}")
print(f"LSTM → Precision: {lstm_precision:.4f}, Recall: {lstm_recall:.4f}, F1: {lstm_f1:.4f}")
print(f"GRU → Precision: {gru_precision:.4f}, Recall: {gru_recall:.4f}, F1: {gru_f1:.4f}")


RNN → Precision: 0.8052, Recall: 0.8646, F1: 0.8338
LSTM → Precision: 0.8522, Recall: 0.9427, F1: 0.8952
GRU → Precision: 0.8718, Recall: 0.8948, F1: 0.8831
